In [1]:
!pip install pydicom

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------------------------- ------------- 1.6/2.4 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 9.7 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Aryan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import pydicom
import numpy as np

In [3]:
def process_patient_ct(patient_folder_path):
    """
    Reads a patient's directory of DICOM files, sorts them into a 3D volume,
    converts to Hounsfield Units (HU), and applies a Lung Window.
    """
    # ---------------------------------------------------------
    # PART A: EXTRACT & SORT
    # ---------------------------------------------------------
    
    # 1. Get all .dcm files in the specific patient folder (e.g., 'C001/')
    dicom_files = [os.path.join(patient_folder_path, f) 
                   for f in os.listdir(patient_folder_path) if f.endswith('.dcm')]
    
    # 2. Read all files into a list
    slices = [pydicom.dcmread(f) for f in dicom_files]
    
    # 3. CRITICAL: Sort slices mathematically by their Z-axis position 
    # (If you don't do this, your 2.5D stacking will mix up random parts of the lungs!)
    slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
    
    # Stack the raw pixel arrays into a single 3D numpy volume
    image_volume = np.stack([s.pixel_array for s in slices])
    
    # ---------------------------------------------------------
    # PART B: HOUNSFIELD UNIT (HU) CONVERSION
    # ---------------------------------------------------------
    # DICOM pixels are stored as raw scanner outputs. We must convert them to standard HU.
    # We grab the Rescale Intercept and Slope from the first slice's metadata.
    intercept = slices[0].RescaleIntercept
    slope = slices[0].RescaleSlope
    
    # Apply the linear conversion formula
    if slope != 1:
        image_volume = slope * image_volume.astype(np.float64)
        image_volume = image_volume.astype(np.int16)
        
    hu_volume = image_volume + np.int16(intercept)
    
    # ---------------------------------------------------------
    # PART C: LUNG WINDOWING
    # ---------------------------------------------------------
    # We apply the specific Lung Window (e.g., -1250 to +250) to isolate lung tissue
    MIN_BOUND = -1250.0
    MAX_BOUND = 250.0
    
    # Clip all values outside this range
    windowed_volume = np.clip(hu_volume, MIN_BOUND, MAX_BOUND)
    
    # Optional: Normalize the windowed volume to a 0.0 to 1.0 range for the neural network
    normalized_volume = (windowed_volume - MIN_BOUND) / (MAX_BOUND - MIN_BOUND)
    
    return normalized_volume

# --- Example Usage ---
# path = f"{dataset_path}/Dataset-S1/COVID-S1/C001"
# patient_3d_volume = process_patient_ct(path)
# print(patient_3d_volume.shape) # Output will be something like (Num_Slices, 512, 512)

In [5]:
# --- 2. The New Batch Processing Function ---
def batch_process_and_save(source_directory, output_directory):
    """
    Loops through all patient folders in a directory, processes them, 
    and saves them as fast-loading .npy files.
    """
    # Create the output folder if it doesn't exist yet
    os.makedirs(output_directory, exist_ok=True)
    
    # Get a list of all items in the source folder (C001, C002, C003...)
    patient_folders = os.listdir(source_directory)
    
    for patient_id in patient_folders:
        patient_path = os.path.join(source_directory, patient_id)
        
        # Check to make sure it's actually a folder and not a hidden file
        if os.path.isdir(patient_path):
            print(f"Processing patient: {patient_id}...")
            
            try:
                # 1. Process the raw DICOMs into a clean 3D volume
                processed_volume = process_patient_ct(patient_path)
                
                # 2. Define exactly where to save it and what to call it
                # Example: "Processed_Data/COVID-S1/C001.npy"
                save_path = os.path.join(output_directory, f"{patient_id}.npy")
                
                # 3. Save the array to your hard drive
                np.save(save_path, processed_volume)
                print(f" --> Saved successfully! Shape: {processed_volume.shape}")
                
            except Exception as e:
                print(f" --> FAILED to process {patient_id}. Error: {e}")

# --- 3. Run the Code based on your folder structure ---
# Point this to where your raw DICOM folders live
raw_data_source = "Dataset-S1/COVID-S1"

# Point this to a brand new folder where you want to store the clean data
clean_data_output = "CovidDataset_Processed/Processed_Data/COVID-S1"

# Execute the loop!
batch_process_and_save(raw_data_source, clean_data_output)

Processing patient: C001...
 --> Saved successfully! Shape: (143, 512, 512)
Processing patient: C002...
 --> Saved successfully! Shape: (144, 512, 512)
Processing patient: C003...
 --> Saved successfully! Shape: (163, 512, 512)
Processing patient: C004...
 --> Saved successfully! Shape: (125, 512, 512)
Processing patient: C005...
 --> Saved successfully! Shape: (122, 512, 512)
Processing patient: C006...
 --> Saved successfully! Shape: (139, 512, 512)
Processing patient: C007...
 --> Saved successfully! Shape: (144, 512, 512)
Processing patient: C008...
 --> Saved successfully! Shape: (122, 512, 512)
Processing patient: C009...
 --> Saved successfully! Shape: (141, 512, 512)
Processing patient: C010...
 --> Saved successfully! Shape: (134, 512, 512)
Processing patient: C011...
 --> Saved successfully! Shape: (139, 512, 512)
Processing patient: C012...
 --> Saved successfully! Shape: (144, 512, 512)
Processing patient: C013...
 --> Saved successfully! Shape: (144, 512, 512)
Processing p